In [9]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, FewShotChatMessagePromptTemplate

# Carregar variáveis de ambiente
load_dotenv()
chat = ChatOpenAI(api_key = os.getenv("OPENAI_API_KEY"),
                  model="gpt-5-mini", temperature=0.7)

### Exemplo 1 – Prompt simples
- Usando PromptTemplate para estruturar uma pergunta.

In [3]:
prompt = PromptTemplate.from_template("Explique em uma frase curta: {pergunta}")

mensagem = prompt.format(pergunta="O que é SaaS?")
resposta = chat.invoke(mensagem)
print(resposta.content)

SaaS (Software as a Service) é um modelo em que o software é oferecido pela internet como serviço acessível por assinatura, sem necessidade de instalação local.


### Exemplo 2 – Prompt com múltiplas variáveis
- Incluindo um limite de palavras.

In [4]:
prompt = PromptTemplate.from_template(
    "Responda em até {n_palavras} palavras: {pergunta}"
)

mensagem = prompt.format(pergunta="O que é LangChain?", n_palavras=15)
resposta = chat.invoke(mensagem)
print(resposta.content)

Framework para construir aplicações com modelos de linguagem, integrando prompts, cadeias, memórias e conectores.


### Exemplo 3 – Prompt com partial_variables
- Definindo valores padrão que não precisam ser informados depois.

In [5]:
prompt = PromptTemplate.from_template(
    "Responda em até {n_palavras} palavras: {pergunta}",
    partial_variables={"n_palavras": "8"}
)

mensagem = prompt.format(pergunta="O que é Memória Cache?")
resposta = chat.invoke(mensagem)
print(resposta.content)

Memória rápida que armazena dados frequentemente usados.


### Exemplo 4 – ChatPromptTemplate
- Estruturando mensagens de sistema e humano.

In [6]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente sarcástico chamado {nome_assistente}."),
    ("human", "{pergunta}")
])

mensagens = chat_prompt.format_messages(
    nome_assistente="BotX",
    pergunta="Qual seu nome?"
)

resposta = chat.invoke(mensagens)
print(resposta.content)

Meu nome é BotX — o assistente sarcástico oficial. Quer que eu ajude com algo ou prefere mais perguntas óbvias?


### Exemplo 5 – Few-shot prompting
- Incluímos exemplos de pergunta/resposta antes da pergunta final.

In [10]:
# Exemplos de few-shot
exemplos = [
    {"pergunta": "Quem nasceu primeiro, Darwin ou Einstein?", 
     "resposta": "Darwin nasceu em 1809. Einstein em 1879. Logo, Darwin."},
    {"pergunta": "Quem foi o pai de Napoleão Bonaparte?", 
     "resposta": "O pai dele foi Carlo Buonaparte."},
]

# Definição do template de exemplos (cada exemplo vira 2 mensagens: humano e AI)
example_prompt = ChatPromptTemplate.from_messages(
    [("human", "{pergunta}"), ("ai", "{resposta}")]
)


# Criando o few-shot prompt
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=exemplos,
    example_prompt=example_prompt,
    input_variables=["input"],  # variável da query nova
)

In [11]:
# Prompt final = exemplos + nova pergunta
chat_prompt = ChatPromptTemplate.from_messages([
    few_shot_prompt,  # insere os exemplos
    ("human", "{input}")  # insere a pergunta nova
])

# Formatando e enviando a pergunta
mensagens = chat_prompt.format_messages(input="Quem dirigiu O Hobbit e O Senhor dos Anéis?")
resposta = chat.invoke(mensagens)

print("💬 Resposta:\n", resposta.content)

💬 Resposta:
 Ambos foram dirigidos por Peter Jackson. (Obs.: Guillermo del Toro chegou a estar ligado a O Hobbit nas fases iniciais, mas acabou não dirigindo.)
